# DB7-016 continuation: 64 missing fits
Same 200ms/10ms TC protocol; no validation; fixed 13 epochs. Original 356 fits excluded. S17/S18 window hashes checked against original. Both T4 GPUs required. Full analysis follows merging with original results.

In [ ]:
from pathlib import Path
import subprocess,sys
SCRIPT_DIR=Path("/kaggle/working/ablation_scripts")
SCRIPT_DIR.mkdir(exist_ok=True)


In [ ]:
(SCRIPT_DIR/'three_branch_model.py').write_text('"""C1: frame-aligned EMG waveform, log-power and inertial encoders for DB7.\n\nThe caller supplies filtered, training-channel-standardized windows in this order:\nEMG (12), ACC (36), gyroscope (36), magnetometer (36). This module does not read\nrecordings, fit the raw input scaler, choose data splits, or filter signals.\n\nBefore training, call ``model.fit_spectral_scaler(training_loader, split=\'train\')``.\nThat loader must contain only already-standardized training windows. Spectral\nstatistics are registered buffers and travel with every model checkpoint.\n"""\n\nfrom __future__ import annotations\n\nimport io as checkpoint_io\nfrom collections.abc import Iterable\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\n\n\nEXPECTED_PARAMETERS = 551_542\nEXPECTED_PARAMETER_BREAKDOWN = {\n    \'waveform\': 81_492,\n    \'spectral\': 67_936,\n    \'inertial\': 123_760,\n    \'fusion\': 41_216,\n    \'temporal\': 197_888,\n    \'attention\': 4_161,\n    \'classifier\': 35_089,\n}\nFRAME_SAMPLES = 200\nFRAME_HOP = 100\nWINDOW_SAMPLES = 400\nN_FRAMES = 3\n\n\ndef unfold_frames(signal: Tensor) -> Tensor:\n    """Return [batch, channels, 3, 200], using only the supplied 400 rows."""\n    if signal.ndim != 3 or signal.shape[-1] != WINDOW_SAMPLES:\n        raise ValueError(f\'Expected [batch, channels, 400], received {tuple(signal.shape)}\')\n    return signal.unfold(-1, FRAME_SAMPLES, FRAME_HOP)\n\n\ndef log_power(z_emg: Tensor, hann: Tensor | None = None) -> Tensor:\n    """Return unscaled log-power [batch, 12, 44, 3] for 20:10:450 Hz.\n\n    ``z_emg`` has already received the fixed training input scaler. Explicit\n    200-sample frames avoid the FFT-length-dependent framing of torch.stft.\n    The FFT is at least float32, including inside an autocast context: CUDA\n    half-precision FFTs cannot implement this non-power-of-two length.\n    """\n    if z_emg.ndim != 3 or z_emg.shape[1:] != (12, WINDOW_SAMPLES):\n        raise ValueError(f\'Expected [batch, 12, 400], received {tuple(z_emg.shape)}\')\n    if not z_emg.is_floating_point():\n        raise TypeError(\'EMG input must be a floating-point tensor\')\n    fft_dtype = torch.float64 if z_emg.dtype == torch.float64 else torch.float32\n    if hann is None:\n        hann = torch.hann_window(FRAME_SAMPLES, periodic=True,\n                                 device=z_emg.device, dtype=fft_dtype)\n    else:\n        hann = hann.to(device=z_emg.device, dtype=fft_dtype)\n        if hann.shape != (FRAME_SAMPLES,):\n            raise ValueError(\'Hann window must contain exactly 200 samples\')\n    with torch.autocast(device_type=z_emg.device.type, enabled=False):\n        frames = unfold_frames(z_emg.to(dtype=fft_dtype))\n        spectrum = torch.fft.rfft(frames * hann, n=FRAME_SAMPLES, dim=-1)\n        power = spectrum.abs().square() / hann.square().sum()\n        # [B, 12, frame, frequency] -> [B, 12, frequency, frame].\n        return torch.log(power[..., 2:46] + 1e-8).permute(0, 1, 3, 2).contiguous()\n\n\ndef _conv_bn_relu(in_channels: int, out_channels: int, kernel: int) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv1d(in_channels, out_channels, kernel, padding=kernel // 2, bias=False),\n        nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),\n        nn.ReLU(),\n    )\n\n\nclass ChannelSqueezeExcitation(nn.Module):\n    def __init__(self, channels: int) -> None:\n        super().__init__()\n        self.gate = nn.Sequential(\n            nn.Linear(channels, channels // 8), nn.ReLU(),\n            nn.Linear(channels // 8, channels), nn.Sigmoid(),\n        )\n\n    def forward(self, x: Tensor) -> Tensor:\n        return x * self.gate(x.mean(-1)).unsqueeze(-1)\n\n\nclass FrameMultiKernelBlock(nn.Module):\n    """Three single-convolution paths, plus projected residual and frame SE."""\n    def __init__(self, in_channels: int, out_channels: int) -> None:\n        super().__init__()\n        self.paths = nn.ModuleList(_conv_bn_relu(in_channels, 32, k) for k in (3, 5, 7))\n        self.merge = nn.Sequential(\n            nn.Conv1d(96, out_channels, 1, bias=False),\n            nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),\n        )\n        self.skip = nn.Conv1d(in_channels, out_channels, 1, bias=False)\n        self.se = ChannelSqueezeExcitation(out_channels)\n\n    def forward(self, x: Tensor) -> Tensor:\n        merged = self.merge(torch.cat([path(x) for path in self.paths], dim=1))\n        return self.se(F.relu(merged + self.skip(x)))\n\n\nclass WaveformEncoder(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        self.stages = nn.Sequential(\n            FrameMultiKernelBlock(12, 64), nn.MaxPool1d(2),\n            FrameMultiKernelBlock(64, 96), nn.MaxPool1d(2),\n        )\n        self.projection = nn.Linear(192, 96)\n\n    def forward(self, frames: Tensor) -> Tensor:\n        # The same encoder processes every frame; batch and frame remain distinct.\n        batch = frames.shape[0]\n        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 12, FRAME_SAMPLES)\n        x = self.stages(x)\n        x = F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))\n        return x.reshape(batch, N_FRAMES, 96).transpose(1, 2).contiguous()\n\n\nclass SpectralEncoder(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        layers: list[nn.Module] = []\n        for incoming, outgoing, kernel, stride in ((12, 32, 5, 2),\n                                                   (32, 64, 5, 2),\n                                                   (64, 96, 3, 1)):\n            layers.extend([\n                nn.Conv2d(incoming, outgoing, (kernel, 1), stride=(stride, 1),\n                          padding=(kernel // 2, 0), bias=False),\n                nn.BatchNorm2d(outgoing, eps=1e-5, momentum=0.1), nn.ReLU(),\n            ])\n        self.stages = nn.Sequential(*layers)\n        self.projection = nn.Conv1d(384, 96, 1, bias=True)\n\n    def forward(self, x: Tensor) -> Tensor:\n        x = self.stages(x)\n        # Preserve four ordered feature-frequency regions, rather than collapsing\n        # absolute frequency location into a single global mean/max vector.\n        regions = torch.stack([x[:, :, begin:end, :].mean(2)\n                               for begin, end in ((0, 3), (3, 6), (6, 9), (9, 11))], dim=2)\n        # Channel-major: each channel retains regions low -> high in adjacent slots.\n        return F.relu(self.projection(regions.flatten(1, 2)))\n\n\nclass InertialModalityEncoder(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        self.stages = nn.Sequential(\n            _conv_bn_relu(36, 48, 5), nn.AvgPool1d(4),\n            _conv_bn_relu(48, 64, 5),\n        )\n        self.projection = nn.Linear(128, 48)\n\n    def forward(self, frames: Tensor) -> Tensor:\n        x = self.stages(frames)\n        return F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))\n\n\nclass InertialEncoder(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        self.modalities = nn.ModuleList(InertialModalityEncoder() for _ in range(3))\n        self.dynamic_projection = nn.Linear(144, 128)\n        self.mean_projection = nn.Linear(108, 128)\n\n    def forward(self, frames: Tensor) -> Tensor:\n        batch = frames.shape[0]\n        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 108, FRAME_SAMPLES)\n        encoded = [encoder(x[:, i * 36:(i + 1) * 36, :])\n                   for i, encoder in enumerate(self.modalities)]\n        dynamic = self.dynamic_projection(torch.cat(encoded, dim=1))\n        static = self.mean_projection(x.mean(-1))\n        # The mean is an additional feature; it is never subtracted from frames.\n        output = F.relu(dynamic + static)\n        return output.reshape(batch, N_FRAMES, 128).transpose(1, 2).contiguous()\n\n\nclass ResidualTemporalBlock(nn.Module):\n    def __init__(self, dilation: int, dropout: float) -> None:\n        super().__init__()\n        self.layers = nn.Sequential(\n            nn.Conv1d(128, 192, 3, dilation=dilation, padding=dilation, bias=False),\n            nn.BatchNorm1d(192, eps=1e-5, momentum=0.1), nn.ReLU(), nn.Dropout(dropout),\n            nn.Conv1d(192, 128, 1, bias=False),\n            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.Dropout(dropout),\n        )\n\n    def forward(self, x: Tensor) -> Tensor:\n        return F.relu(x + self.layers(x))\n\n\nclass ThreeBranchC1(nn.Module):\n    """Exact 551,542-parameter C1, compatible with Trainer\'s model(x) call."""\n    def __init__(self, dropout: float = 0.15) -> None:\n        super().__init__()\n        self.waveform = WaveformEncoder()\n        self.spectral = SpectralEncoder()\n        self.inertial = InertialEncoder()\n        self.fusion = nn.Sequential(\n            nn.Conv1d(320, 128, 1, bias=False),\n            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.ReLU(),\n        )\n        self.temporal = nn.Sequential(ResidualTemporalBlock(1, dropout),\n                                      ResidualTemporalBlock(2, dropout))\n        self.attention = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))\n        self.classifier = nn.Sequential(nn.Linear(256, 128), nn.ReLU(),\n                                        nn.Dropout(dropout), nn.Linear(128, 17))\n        self.register_buffer(\'hann\', torch.hann_window(FRAME_SAMPLES, periodic=True))\n        self.register_buffer(\'spectral_mean\', torch.zeros(12, 44))\n        self.register_buffer(\'spectral_std\', torch.ones(12, 44))\n        self.register_buffer(\'spectral_constant\', torch.zeros(12, 44, dtype=torch.bool))\n        self.register_buffer(\'spectral_fitted\', torch.tensor(False))\n        self.register_buffer(\'spectral_fit_frames\', torch.tensor(0, dtype=torch.long))\n        if self.count_params() != EXPECTED_PARAMETERS:\n            raise AssertionError(f\'C1 parameter mismatch: {self.count_params()}\')\n\n    def count_params(self) -> int:\n        return sum(parameter.numel() for parameter in self.parameters())\n\n    def parameter_breakdown(self) -> dict[str, int]:\n        return {name: sum(parameter.numel() for parameter in getattr(self, name).parameters())\n                for name in EXPECTED_PARAMETER_BREAKDOWN}\n\n    @torch.no_grad()\n    def fit_spectral_scaler(self, training_batches: Iterable[Any], *, split: str = \'train\',\n                            std_floor: float = 1e-6) -> dict[str, Any]:\n        """Fit frozen moments without running a CNN or updating BatchNorm.\n\n        The iterable yields standardized x or (x,y) batches. One observation for\n        each channel/frequency bin is one frame; all three frames of every\n        training window receive equal weight. Overlap is intentional. Population\n        variance uses a stable float64 batched merge; SD below ``std_floor`` is\n        flagged and clamped. A second fit is rejected to avoid accidental reuse\n        on validation/test; create a fresh model for a new subject or fold.\n\n        The explicit split guard cannot establish arbitrary generator provenance.\n        The caller must save source/window hashes. A DataLoader exposing a dataset\n        with a ``meta[\'split\']`` column receives an additional provenance check.\n        """\n        if split != \'train\':\n            raise ValueError(\'Spectral scaler may be fitted only on the train split\')\n        if bool(self.spectral_fitted.item()):\n            raise RuntimeError(\'Spectral scaler is already fitted; use a fresh model for another fold\')\n        if std_floor <= 0:\n            raise ValueError(\'std_floor must be positive\')\n        dataset = getattr(training_batches, \'dataset\', None)\n        metadata = getattr(dataset, \'meta\', None)\n        if metadata is not None and \'split\' in metadata:\n            if set(metadata[\'split\'].unique()) != {\'train\'}:\n                raise ValueError(\'Spectral fitting DataLoader contains non-training metadata\')\n        device = self.spectral_mean.device\n        count = 0\n        mean = torch.zeros(12, 44, dtype=torch.float64, device=device)\n        m2 = torch.zeros_like(mean)\n        for batch in training_batches:\n            x = batch[0] if isinstance(batch, (tuple, list)) else batch\n            x = torch.as_tensor(x, device=device, dtype=self.spectral_mean.dtype)\n            if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES) or x.shape[0] == 0:\n                raise ValueError(\'Spectral scaler requires nonempty [B,120,400] training batches\')\n            if not bool(torch.isfinite(x).all().item()):\n                raise ValueError(\'Nonfinite training input in spectral scaler fit\')\n            values = log_power(x[:, :12], self.hann).to(torch.float64)\n            batch_count = values.shape[0] * N_FRAMES\n            batch_mean = values.mean(dim=(0, 3))\n            batch_m2 = (values - batch_mean[None, :, :, None]).square().sum(dim=(0, 3))\n            combined_count = count + batch_count\n            delta = batch_mean - mean\n            m2 += batch_m2 + delta.square() * (count * batch_count / combined_count)\n            mean += delta * (batch_count / combined_count)\n            count = combined_count\n        if not count:\n            raise ValueError(\'Cannot fit the spectral scaler on an empty iterable\')\n        std = (m2 / count).clamp_min(0).sqrt()\n        self.spectral_mean.copy_(mean)\n        self.spectral_constant.copy_(std < std_floor)\n        self.spectral_std.copy_(std.clamp_min(std_floor))\n        self.spectral_fit_frames.fill_(count)\n        self.spectral_fitted.fill_(True)\n        return {\n            \'split\': \'train\', \'windows\': count // N_FRAMES, \'frames\': count,\n            \'statistic\': \'float64_population_moments_over_training_windows_and_three_frames\',\n            \'std_floor\': float(std_floor),\n            \'constant_bins\': int(self.spectral_constant.sum().item()),\n            \'mean\': self.spectral_mean.detach().cpu().tolist(),\n            \'std\': self.spectral_std.detach().cpu().tolist(),\n            \'constant\': self.spectral_constant.detach().cpu().tolist(),\n        }\n\n    def forward_features(self, x: Tensor, *, retain_sequences: bool = False) -> dict[str, Any]:\n        if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES):\n            raise ValueError(f\'C1 expects [B,120,400], received {tuple(x.shape)}\')\n        if not bool(self.spectral_fitted.item()):\n            raise RuntimeError(\'Fit the spectral scaler using training windows before model(x)\')\n        frames = unfold_frames(x)\n        waveform = self.waveform(frames[:, :12])\n        power = log_power(x[:, :12], self.hann)\n        scaled_power = ((power - self.spectral_mean[None, :, :, None]) /\n                        self.spectral_std[None, :, :, None])\n        spectral = self.spectral(scaled_power)\n        inertial = self.inertial(frames[:, 12:])\n        fused = self.temporal(self.fusion(torch.cat([waveform, spectral, inertial], dim=1)))\n        scores = self.attention(fused.transpose(1, 2)).squeeze(-1)\n        attention = torch.softmax(scores, dim=-1)\n        weighted = (fused * attention[:, None, :]).sum(-1)\n        embedding = torch.cat([weighted, fused.mean(-1)], dim=1)\n        result = {\n            \'logits\': self.classifier(embedding), \'embedding\': embedding,\n            \'branch_embeddings\': {\'waveform\': waveform.mean(-1),\n                                  \'spectral\': spectral.mean(-1),\n                                  \'inertial\': inertial.mean(-1)},\n            \'attention\': attention,\n        }\n        if retain_sequences:\n            result[\'sequences\'] = {\'waveform\': waveform, \'spectral\': spectral,\n                                   \'inertial\': inertial, \'fused\': fused}\n        return result\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.forward_features(x)[\'logits\']\n\n\ndef model_preflight(device: str | torch.device = \'cpu\') -> dict[str, Any]:\n    """Meaningful synthetic checks for the installed Kaggle PyTorch runtime.\n\n    Does not touch recordings or disk, install packages, or start an experiment.\n    CPU and applicable CUDA RNG state are restored before return.\n    """\n    target = torch.device(device)\n    # torch.manual_seed also seeds CUDA generators. Preserve every initialized\n    # device\'s stream instead of changing an unused second Kaggle GPU\'s RNG.\n    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []\n    with torch.random.fork_rng(devices=cuda_devices):\n        torch.manual_seed(941)\n        model = ThreeBranchC1().to(target)\n        assert model.parameter_breakdown() == EXPECTED_PARAMETER_BREAKDOWN\n        x = torch.randn(3, 120, WINDOW_SAMPLES, device=target)\n        try:\n            model(x)\n        except RuntimeError as error:\n            assert \'spectral scaler\' in str(error)\n        else:\n            raise AssertionError(\'Unfitted spectral scaling must block model inference\')\n        try:\n            model.fit_spectral_scaler([x], split=\'test\')\n        except ValueError as error:\n            assert \'train split\' in str(error)\n        else:\n            raise AssertionError(\'The spectral scaler must reject non-training splits\')\n        frames = unfold_frames(x)\n        assert frames.shape == (3, 120, 3, 200)\n        for frame in range(N_FRAMES):\n            torch.testing.assert_close(frames[:, :, frame], x[:, :, frame * 100:frame * 100 + 200])\n        # Independent explicit-frame transform verifies frequency/frame ordering.\n        hann = torch.hann_window(200, periodic=True, device=target)\n        reference = torch.stack([\n            torch.log(torch.fft.rfft(x[:, :12, begin:begin + 200] * hann, dim=-1)\n                      .abs().square()[..., 2:46] / hann.square().sum() + 1e-8)\n            for begin in range(0, 201, 100)], dim=-1)\n        torch.testing.assert_close(log_power(x[:, :12]), reference)\n        before_bn = {name: value.clone() for name, value in model.named_buffers()\n                     if \'running_\' in name or \'num_batches_tracked\' in name}\n        statistics = model.fit_spectral_scaler([(x[:2], torch.zeros(2)),\n                                                (x[2:], torch.zeros(1))], split=\'train\')\n        assert statistics[\'windows\'] == 3 and statistics[\'frames\'] == 9\n        try:\n            model.fit_spectral_scaler([x], split=\'train\')\n        except RuntimeError as error:\n            assert \'already fitted\' in str(error)\n        else:\n            raise AssertionError(\'A fitted spectral scaler must reject accidental refitting\')\n        reference_double = reference.double()\n        expected_mean = reference_double.mean(dim=(0, 3))\n        expected_std = reference_double.permute(1, 2, 0, 3).reshape(12, 44, -1).std(-1, correction=0)\n        torch.testing.assert_close(model.spectral_mean, expected_mean.float(), rtol=2e-5, atol=2e-6)\n        torch.testing.assert_close(model.spectral_std, expected_std.float(), rtol=2e-5, atol=2e-6)\n        for name, value in model.named_buffers():\n            if name in before_bn:\n                torch.testing.assert_close(value, before_bn[name], rtol=0, atol=0)\n        frozen_scaler = {name: value.clone() for name, value in model.named_buffers()\n                         if name.startswith(\'spectral_\')}\n        model.train()\n        outputs = model.forward_features(x, retain_sequences=True)\n        assert outputs[\'logits\'].shape == (3, 17)\n        assert outputs[\'embedding\'].shape == (3, 256)\n        assert outputs[\'attention\'].shape == (3, 3)\n        for branch, width in ((\'waveform\', 96), (\'spectral\', 96), (\'inertial\', 128), (\'fused\', 128)):\n            assert outputs[\'sequences\'][branch].shape == (3, width, 3)\n        torch.testing.assert_close(outputs[\'attention\'].sum(-1), torch.ones(3, device=target))\n        loss = F.cross_entropy(outputs[\'logits\'], torch.tensor([0, 8, 16], device=target))\n        loss.backward()\n        for name, module in ((\'waveform\', model.waveform), (\'spectral\', model.spectral),\n                             (\'acc\', model.inertial.modalities[0]),\n                             (\'gyro\', model.inertial.modalities[1]),\n                             (\'mag\', model.inertial.modalities[2]),\n                             (\'inertial_mean\', model.inertial.mean_projection),\n                             (\'fusion\', model.fusion)):\n            gradients = [p.grad for p in module.parameters() if p.grad is not None]\n            assert gradients and all(bool(torch.isfinite(g).all().item()) for g in gradients), name\n            assert any(bool(g.abs().sum().item() > 0) for g in gradients), name\n        for name, value in model.named_buffers():\n            if name in frozen_scaler:\n                torch.testing.assert_close(value, frozen_scaler[name], rtol=0, atol=0)\n        model.eval()\n        with torch.no_grad():\n            expected = model(x)\n            # An input to another branch cannot change waveform/spectral tokens.\n            changed = x.clone()\n            changed[:, 12:] += 1.75\n            original_features = model.forward_features(x, retain_sequences=True)\n            altered_features = model.forward_features(changed, retain_sequences=True)\n            for name in (\'waveform\', \'spectral\'):\n                torch.testing.assert_close(original_features[\'sequences\'][name],\n                                           altered_features[\'sequences\'][name], rtol=0, atol=0)\n            checkpoint = checkpoint_io.BytesIO()\n            torch.save(model.state_dict(), checkpoint)\n            checkpoint.seek(0)\n            restored = ThreeBranchC1().to(target)\n            restored.load_state_dict(torch.load(checkpoint, map_location=target, weights_only=True))\n            restored.eval()\n            torch.testing.assert_close(restored(x), expected, rtol=0, atol=0)\n        return {\'success\': True, \'parameters\': model.count_params(),\n                \'parameter_breakdown\': model.parameter_breakdown(), \'device\': str(target),\n                \'torch_version\': torch.__version__, \'frames\': 3, \'frequency_bins\': 44,\n                \'output_shape\': list(expected.shape),\n                \'checks\': [\'exact_parameter_count\', \'frame_alignment\', \'explicit_fft_reference\',\n                           \'unfitted_nontraining_and_refit_guards\',\n                           \'training_spectral_moments\', \'scaler_fit_does_not_update_batchnorm\',\n                           \'frozen_scaler\', \'forward_backward_all_branches\',\n                           \'branch_input_isolation\', \'checkpoint_round_trip\']}\n\n\nif __name__ == \'__main__\':\n    import json\n    print(json.dumps(model_preflight(\'cuda\' if torch.cuda.is_available() else \'cpu\'), indent=2))\n',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'ablation_model.py').write_text('"""Retrained branch subsets. Full WSI is numerically identical to original C1."""\nimport torch\nfrom torch import nn\nfrom three_branch_model import ThreeBranchC1, unfold_frames, log_power\n\nARMS=(\'W\',\'S\',\'I\',\'WS\',\'WI\',\'SI\',\'WSI\')\nNAMES={\'W\':\'waveform\',\'S\':\'spectral\',\'I\':\'inertial\'}\nSLICES={\'W\':(0,96),\'S\':(96,192),\'I\':(192,320)}\n\nclass AblationC1(ThreeBranchC1):\n    def __init__(self,arm,dropout=.15):\n        assert arm in ARMS\n        super().__init__(dropout)\n        self.arm=arm\n        self.active_names=[NAMES[k] for k in arm]\n        indices=[i for k in arm for i in range(*SLICES[k])]\n        if arm!=\'WSI\':\n            old=self.fusion[0]\n            # Preserve the shared initialization; restore RNG after constructor.\n            with torch.random.fork_rng():\n                replacement=nn.Conv1d(len(indices),128,1,bias=False)\n            with torch.no_grad():replacement.weight.copy_(old.weight[:,indices])\n            self.fusion[0]=replacement\n            for k,name in NAMES.items():\n                if k not in arm:delattr(self,name)\n\n    def forward_features(self,x,retain_sequences=False):\n        frames=unfold_frames(x);seq={}\n        if \'W\' in self.arm:seq[\'waveform\']=self.waveform(frames[:,:12])\n        if \'S\' in self.arm:\n            if not self.spectral_fitted:raise RuntimeError(\'Fit spectral scaler first\')\n            p=log_power(x[:,:12],self.hann)\n            seq[\'spectral\']=self.spectral((p-self.spectral_mean[None,:,:,None])/self.spectral_std[None,:,:,None])\n        if \'I\' in self.arm:seq[\'inertial\']=self.inertial(frames[:,12:])\n        fused=self.temporal(self.fusion(torch.cat(list(seq.values()),dim=1)))\n        attention=self.attention(fused.transpose(1,2)).squeeze(-1).softmax(-1)\n        embedding=torch.cat([(fused*attention[:,None]).sum(-1),fused.mean(-1)],dim=1)\n        out=dict(logits=self.classifier(embedding),embedding=embedding,\n                 attention=attention,branch_embeddings={k:v.mean(-1) for k,v in seq.items()})\n        if retain_sequences:out[\'sequences\']={**seq,\'fused\':fused}\n        return out\n\ndef preflight():\n    """Exact full-model parity; gradient coverage and excluded-input isolation."""\n    x=torch.randn(3,120,400,device=\'cuda\')\n    torch.manual_seed(77);ref=ThreeBranchC1().cuda()\n    ref.fit_spectral_scaler([x]);ref.eval()\n    torch.manual_seed(77);full=AblationC1(\'WSI\').cuda()\n    full.load_state_dict(ref.state_dict());full.eval()\n    torch.testing.assert_close(full(x),ref(x),rtol=0,atol=0)\n    rows=[]\n    for arm in ARMS:\n        model=AblationC1(arm).cuda()\n        if \'S\' in arm:model.fit_spectral_scaler([x])\n        model.train();out=model(x)\n        nn.functional.cross_entropy(out,torch.tensor([0,8,16],device=\'cuda\')).backward()\n        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())\n        model.eval()\n        changed=x.clone()\n        if \'I\' not in arm:changed[:,12:]+=10\n        if arm==\'I\':changed[:,:12]+=10\n        torch.testing.assert_close(model(x),model(changed),rtol=0,atol=0)\n        rows.append(dict(arm=arm,parameters=model.count_params()))\n    return rows\n',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'tc_support.py').write_text('"""C1 under TC-AiFusion\'s split and training budget, retaining all inertial sensors.\n\nNo TC feature images or external-window history are used: these would replace C1.\nAll preprocessing fits use training repetitions only. Test is evaluated once,\nafter the fixed final epoch. This is annotation-assisted offline classification.\n"""\nimport gc, hashlib, json, random, time, traceback, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.io import loadmat\nfrom scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom three_branch_model import ThreeBranchC1, model_preflight\n\n\nclass Settings:\n    SUBJECTS = list(range(1, 21))\n    TRAIN_REPS = [1, 3, 4, 6]\n    TEST_REPS = [2, 5]\n    FS = 2000\n    WINDOW = 400  # 200 ms\n    STEP = 20     # 10 ms\n    BATCH_SIZE = 512\n    EPOCHS = 13\n    DROPOUT = 0.65  # Match TC-AiFusion\'s training regularization setting.\n    SEED = 42\n    SMOKE = False\n    INPUT = None\n    OUTPUT = Path(\'/kaggle/working\') if Path(\'/kaggle\').exists() else Path.cwd()\n    AUTOMATION = {}\n\n\ndef write_json(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value,indent=2),encoding=\'utf-8\')\n\n\ndef find_input():\n    if Settings.INPUT is not None:return Path(Settings.INPUT)\n    for candidate in sorted(Path(\'/kaggle/input\').rglob(\'Subject_1\')):\n        if candidate.is_dir():return candidate.parent\n    raise FileNotFoundError(\'Attach rayaanraza1/ninapro-db7 or set Settings.INPUT to the subject-folder root.\')\n\n\ndef aligned_interval(array, emg_length, start, end):\n    """TC alignment rule, independently applied to ACC, gyro and magnetometer."""\n    if len(array)==emg_length:return array[start:end].astype(np.float32,copy=True)\n    ratio=len(array)/float(emg_length)\n    a0=max(0,min(int(np.floor(start*ratio)),len(array)-1))\n    a1=max(a0+1,min(int(np.ceil(end*ratio)),len(array)))\n    segment=array[a0:a1];length=end-start\n    divisor=np.gcd(len(segment),length)\n    result=resample_poly(segment,length//divisor,len(segment)//divisor,axis=0).astype(np.float32)\n    if len(result)<length:result=np.vstack([result,np.repeat(result[-1:],length-len(result),axis=0)])\n    return result[:length]\n\n\ndef load_subject(root, subject, output):\n    directory=root/f\'Subject_{subject}\'\n    if not directory.exists():directory=root/f\'S{subject}\'\n    files=sorted(directory.rglob(\'*_E1_*.mat\'))\n    if len(files)!=1:raise ValueError(f\'S{subject}: expected one E1 file, found {len(files)}\')\n    data=loadmat(files[0],variable_names=[\'emg\',\'acc\',\'gyro\',\'mag\',\'restimulus\',\'rerepetition\',\'subject\',\'exercise\'])\n    def sensor(key,channels):\n        x=np.asarray(data[key],dtype=np.float32)\n        if x.ndim!=2:raise ValueError(f\'{key}: not 2D\')\n        if x.shape[1]!=channels and x.shape[0]==channels:x=x.T\n        if x.shape[1]!=channels or not np.isfinite(x).all():raise ValueError(f\'{key}: invalid channels/values\')\n        return x\n    emg=sensor(\'emg\',12)\n    inertial=[sensor(key,36) for key in [\'acc\',\'gyro\',\'mag\']]\n    labels=np.asarray(data[\'restimulus\']).reshape(-1).astype(int)\n    reps=np.asarray(data[\'rerepetition\']).reshape(-1).astype(int)\n    assert len(emg)==len(labels)==len(reps), \'Reject silent label/signal truncation\'\n    assert set(np.unique(labels))==set(range(18))\n    edges=np.r_[0,np.flatnonzero(np.diff(labels))+1,len(labels)]\n    sos=butter(4,[20,450],btype=\'bandpass\',fs=Settings.FS,output=\'sos\')\n    b,a=iirnotch(50,30,fs=Settings.FS)\n    segments=[];records=[]\n    for start,end in zip(edges[:-1],edges[1:]):\n        gesture=int(labels[start])\n        if gesture==0:continue\n        native=np.unique(reps[start:end])\n        if len(native)!=1 or native[0] not in range(1,7):\n            raise ValueError(\'Gesture run must contain exactly one valid repetition; do not assign by majority\')\n        repetition=int(native[0]);split=\'train\' if repetition in Settings.TRAIN_REPS else \'test\'\n        filtered=sosfiltfilt(sos,emg[start:end],axis=0)\n        filtered=filtfilt(b,a,filtered,axis=0).astype(np.float32)\n        x=np.concatenate([filtered]+[aligned_interval(v,len(emg),int(start),int(end)) for v in inertial],axis=1)\n        if not np.isfinite(x).all():raise ValueError(\'Nonfinite filtered/aligned signal\')\n        segments.append(x)\n        records.append(dict(subject=subject,gesture=gesture,native_repetition=repetition,split=split,\n            run_start=int(start),run_end=int(end),file_name=files[0].name,segment=len(segments)-1))\n    inventory=pd.DataFrame(records)\n    for gesture in range(1,18):\n        rows=inventory[inventory.gesture==gesture]\n        assert len(rows)==6 and set(rows.native_repetition)==set(range(1,7))\n    # TC normalization: every sample of each active training segment counted once.\n    total=np.zeros(120,np.float64);squares=total.copy();count=0\n    for rec in records:\n        if rec[\'split\']==\'train\':\n            x=segments[rec[\'segment\']].astype(np.float64)\n            total+=x.sum(0);squares+=np.square(x).sum(0);count+=len(x)\n    mean=total/count;variance=np.maximum(squares/count-mean*mean,1e-12)\n    std=np.sqrt(variance)\n    np.savez_compressed(output/\'input_scaler.npz\',mean=mean.astype(np.float32),std=std.astype(np.float32),training_sample_count=count)\n    inventory.to_csv(output/\'repetition_inventory.csv\',index=False)\n    write_json(output/\'identity.json\',dict(folder_subject=subject,internal_subject=int(np.asarray(data.get(\'subject\',subject)).item()),\n        file=files[0].name,emg_sha256=hashlib.sha256(emg.tobytes()).hexdigest(),\n        sensor_shapes={key:list(value.shape) for key,value in zip([\'emg\',\'acc\',\'gyro\',\'mag\'],[emg]+inertial)},\n        alignment_fallback={key:len(value)!=len(emg) for key,value in zip([\'acc\',\'gyro\',\'mag\'],inertial)}))\n    return segments,inventory,mean.astype(np.float32),std.astype(np.float32)\n\n\nclass TCWindows(Dataset):\n    def __init__(self,segments,inventory,mean,std,split):\n        self.segments=segments;self.mean=mean;self.std=std\n        rows=[]\n        for rec in inventory[inventory.split==split].to_dict(\'records\'):\n            for offset in range(0,rec[\'run_end\']-rec[\'run_start\']-Settings.WINDOW+1,Settings.STEP):\n                start=rec[\'run_start\']+offset;end=start+Settings.WINDOW\n                phase=((start+end)/2-rec[\'run_start\'])/(rec[\'run_end\']-rec[\'run_start\'])\n                rows.append(dict(**rec,offset=offset,window_start=start,window_end=end,\n                    phase_fraction=phase,phase=\'early\' if phase<1/3 else \'middle\' if phase<2/3 else \'late\'))\n        self.meta=pd.DataFrame(rows)\n        self.y=self.meta.gesture.to_numpy()-1\n        self.locations=self.meta[[\'segment\',\'offset\']].to_numpy()\n        assert not self.meta.duplicated([\'subject\',\'gesture\',\'native_repetition\',\'window_start\']).any()\n    def __len__(self):return len(self.y)\n    def __getitem__(self,index):\n        seg,start=self.locations[index]\n        x=self.segments[seg][start:start+Settings.WINDOW]\n        return torch.from_numpy(((x-self.mean)/self.std).T.copy()),int(self.y[index])\n\n\ndef set_seed(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\n\ndef evaluate(model,dataset,device,path):\n    path.mkdir(exist_ok=True)\n    model.eval();probs=[];embeddings=[];attention=[]\n    branch={key:[] for key in model.active_names}\n    with torch.no_grad():\n        for x,y in DataLoader(dataset,batch_size=Settings.BATCH_SIZE,shuffle=False):\n            out=model.forward_features(x.to(device))\n            probs.append(out[\'logits\'].softmax(1).cpu().numpy());embeddings.append(out[\'embedding\'].cpu().numpy())\n            attention.append(out[\'attention\'].cpu().numpy())\n            for key in branch:branch[key].append(out[\'branch_embeddings\'][key].cpu().numpy())\n    p=np.concatenate(probs);pred=p.argmax(1)\n    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)\n    frame=dataset.meta.copy();frame[\'seed_base\']=Settings.SEED;frame[\'arm\']=model.arm;frame[\'prediction\']=pred+1;frame[\'correct\']=pred==dataset.y;frame[\'confidence\']=p.max(1)\n    if path.name == \'test\':\n        frame.to_csv(path/\'predictions.csv\',index=False)\n        arrays=dict(y_true=dataset.y, probabilities=p)\n        if Settings.SEED==42:\n            arrays.update(embeddings=np.concatenate(embeddings),attention=np.concatenate(attention),\n                          **{k:np.concatenate(v) for k,v in branch.items()})\n        np.savez_compressed(path/\'probabilities_embeddings.npz\',**arrays)\n    errors=frame.groupby([\'subject\',\'gesture\',\'native_repetition\']).agg(windows=(\'correct\',\'size\'),correct=(\'correct\',\'sum\')).reset_index()\n    errors[\'wrong\']=errors.windows-errors.correct;errors[\'error_percent\']=100*errors.wrong/errors.windows\n    errors.to_csv(path/\'gesture_errors.csv\',index=False)\n    phase=frame.groupby([\'gesture\',\'native_repetition\',\'phase\']).agg(windows=(\'correct\',\'size\'),correct=(\'correct\',\'sum\')).reset_index()\n    phase[\'wrong\']=phase.windows-phase.correct;phase.to_csv(path/\'phase_errors.csv\',index=False)\n    confusion=pd.crosstab(frame.gesture,frame.prediction).reindex(index=range(1,18),columns=range(1,18),fill_value=0)\n    confusion.to_csv(path/\'confusion.csv\')\n    tp=np.diag(confusion);den=confusion.sum(axis=0).to_numpy()+confusion.sum(axis=1).to_numpy()\n    f1=2*tp/np.maximum(den,1)\n    metrics=dict(accuracy=float(frame.correct.mean()),macro_f1=float(f1.mean()),n_windows=len(frame),\n        wrong=int((~frame.correct).sum()),nll=float(-np.log(p[np.arange(len(p)),dataset.y].clip(1e-12)).mean()))\n    write_json(path/\'metrics.json\',metrics)\n    return metrics\n\n\n',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'tc_worker.py').write_text('"""Seven retrained subsets using the previous TC protocol, two isolated GPUs."""\nimport sys,os,gc,time,traceback,hashlib,json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\nfrom ablation_model import ARMS,AblationC1,preflight\nimport tc_support as T\nKEYS=[\'subject\',\'gesture\',\'native_repetition\',\'window_start\',\'window_end\']\n\ndef features(ds,folder):\n    rows=[]\n    for start in range(0,len(ds),128):\n        x=np.stack([ds.segments[seg][offset:offset+400].T for seg,offset in ds.locations[start:start+128]]).astype(np.float64)\n        emg=x[:,:12];p=abs(np.fft.rfft(emg*np.hanning(400),axis=-1))**2\n        hz=np.fft.rfftfreq(400,1/2000);mask=(hz>=20)&(hz<=450);power=p[:,:,mask];values={}\n        descriptors={\'rms200\':np.sqrt((emg**2).mean(-1)),\'mav200\':abs(emg).mean(-1),\n          \'waveform_length\':abs(np.diff(emg,axis=-1)).sum(-1),\n          \'mean_frequency\':(power*hz[mask]).sum(-1)/np.maximum(power.sum(-1),1e-30)}\n        for name,v in descriptors.items():\n            for c in range(12):values[f\'emg_{c+1:02}_{name}\']=v[:,c]\n        for name,a in [(\'acc\',12),(\'gyro\',48),(\'mag\',84)]:\n            for stat,v in [(\'mean\',x[:,a:a+36].mean(-1)),(\'std\',x[:,a:a+36].std(-1))]:\n                for c in range(36):values[f\'{name}_{c+1:02}_{stat}\']=v[:,c]\n        rows.append(pd.DataFrame(values))\n    folder.mkdir(exist_ok=True)\n    pd.concat([ds.meta[KEYS+[\'phase\']].reset_index(drop=True),pd.concat(rows,ignore_index=True)],axis=1).to_csv(folder/\'input_features.csv\',index=False)\n\ndef fit(arm,seed,subject,train,test,folder,epochs):\n    folder.mkdir(parents=True,exist_ok=True);actual=seed+1000*subject\n    T.Settings.SEED=seed;T.set_seed(actual);model=AblationC1(arm,T.Settings.DROPOUT).cuda()\n    if \'S\' in arm:T.write_json(folder/\'spectral_scaler.json\',model.fit_spectral_scaler(DataLoader(train,batch_size=512,shuffle=False)))\n    T.set_seed(actual)\n    loader=DataLoader(train,batch_size=512,shuffle=True,generator=torch.Generator().manual_seed(actual+12345))\n    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=0)\n    history=[];torch.cuda.reset_peak_memory_stats()\n    for epoch in range(1,epochs+1):\n        lr=1e-3 if epoch<=3 else 1e-4 if epoch<=9 else 1e-5\n        for group in opt.param_groups:group[\'lr\']=lr\n        model.train();total=correct=0;loss_sum=0.;tick=time.perf_counter()\n        for x,y in loader:\n            x=x.cuda();y=y.cuda();opt.zero_grad(set_to_none=True)\n            logits=model(x);loss=F.cross_entropy(logits,y)\n            if not torch.isfinite(loss):raise ValueError(\'Nonfinite training loss\')\n            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5);opt.step()\n            total+=len(y);correct+=int((logits.argmax(1)==y).sum());loss_sum+=float(loss.detach())*len(y)\n        history.append(dict(epoch=epoch,lr=lr,train_loss=loss_sum/total,train_accuracy=correct/total,seconds=time.perf_counter()-tick))\n        pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(f\'GPU{os.environ["CUDA_VISIBLE_DEVICES"]} S{subject:02} seed{seed} {arm} epoch{epoch}/{epochs} train={correct/total:.4f}\',flush=True)\n    if seed==42:torch.save(model.state_dict(),folder/\'final_model.pt\')\n    metrics=[]\n    for split,ds in [(\'train\',train),(\'test\',test)]:\n        m=T.evaluate(model,ds,\'cuda\',folder/split);m[\'f1_macro\']=m.pop(\'macro_f1\')\n        metrics.append(dict(subject=subject,seed_base=seed,arm=arm,split=split,**m))\n    pd.DataFrame(metrics).to_csv(folder/\'metrics.csv\',index=False)\n    model.eval();x=train[0][0][None].cuda()\n    with torch.no_grad():\n        for _ in range(10):model(x)\n        torch.cuda.synchronize();lat=[]\n        for _ in range(50):\n            t=time.perf_counter();model(x);torch.cuda.synchronize();lat.append((time.perf_counter()-t)*1000)\n    T.write_json(folder/\'fit_manifest.json\',dict(success=True,subject=subject,seed_base=seed,arm=arm,epochs=epochs,\n        parameters=model.count_params(),validation=None,checkpoint=\'fixed final epoch\',\n        window_hashes={s:hashlib.sha256(d.meta[KEYS].to_csv(index=False).encode()).hexdigest() for s,d in [(\'train\',train),(\'test\',test)]},\n        inference_median_ms=float(np.median(lat)),inference_p95_ms=float(np.quantile(lat,.95)),\n        timing_scope=\'batch-one forward only; shared host load; not end-to-end latency\',peak_cuda_memory_bytes=torch.cuda.max_memory_allocated()))\n    del model,opt,loader;gc.collect();torch.cuda.empty_cache()\n\ndef main():\n    root=Path(sys.argv[1]);gpu=int(sys.argv[2]);smoke=sys.argv[3]==\'smoke\'\n    assert torch.cuda.device_count()==1\n    T.write_json(root/f\'gpu{gpu}_preflight.json\',dict(gpu=gpu,pid=os.getpid(),name=torch.cuda.get_device_name(0),checks=preflight()))\n    print(f\'GPU{gpu} PREFLIGHT PASSED\',flush=True);source=T.find_input()\n    pending=json.loads(Path(\'remaining.json\').read_text())\n    subjects=sorted({s for s,seed,arm in pending if (s-1)%2==gpu})\n    for subject in subjects:\n        sf=root/f\'S{subject:02}\';sf.mkdir(parents=True,exist_ok=True)\n        segments,inventory,mean,std=T.load_subject(source,subject,sf)\n        train=T.TCWindows(segments,inventory,mean,std,\'train\');test=T.TCWindows(segments,inventory,mean,std,\'test\')\n        expected=json.loads(Path(\'expected_hashes.json\').read_text()).get(str(subject))\n        if expected:\n            assert expected=={s:hashlib.sha256(d.meta[KEYS].to_csv(index=False).encode()).hexdigest() for s,d in [(\'train\',train),(\'test\',test)]}, \'Window mismatch against original run\'\n        for split,ds in [(\'train\',train),(\'test\',test)]:\n            ds.meta.to_csv(sf/f\'{split}_windows.csv\',index=False);features(ds,sf/split)\n        for seed in ([42] if smoke else [42,43,44]):\n            for arm in ARMS:\n                if [subject,seed,arm] not in pending:continue\n                fit(arm,seed,subject,train,test,sf/f\'seed_{seed}\'/arm,2 if smoke else 13)\n                print(f\'GPU{gpu} COMPLETE S{subject:02} seed{seed} {arm}\',flush=True)\n        del train,test,segments,ds;gc.collect();torch.cuda.empty_cache()\n    T.write_json(root/f\'worker_{gpu}_complete.json\',dict(success=True))\n\nif __name__==\'__main__\':\n    try:main()\n    except Exception:\n        (Path(sys.argv[1])/f\'gpu{sys.argv[2]}_failure.txt\').write_text(traceback.format_exc());raise\n',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'launch.py').write_text('"""Dual-GPU smoke gate, followed by 66 fits and diagnostic reduction."""\nimport os,sys,json,time,subprocess,zipfile,traceback\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\ndef run_pair(root,mode):\n    root.mkdir(parents=True,exist_ok=True);processes=[];handles=[]\n    try:\n        for gpu in (0,1):\n            env=os.environ.copy();env.update(CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',PYTHONUNBUFFERED=\'1\')\n            handle=open(root/f\'gpu{gpu}.log\',\'w\');handles.append(handle)\n            processes.append(subprocess.Popen([sys.executable,\'tc_worker.py\',str(root),str(gpu),mode],env=env,stdout=handle,stderr=subprocess.STDOUT))\n        offsets=[0,0];last_telemetry=0\n        while True:\n            for gpu in (0,1):\n                with open(root/f\'gpu{gpu}.log\') as stream:\n                    stream.seek(offsets[gpu]);chunk=stream.read();offsets[gpu]=stream.tell()\n                if chunk:print(chunk,end=\'\',flush=True)\n            codes=[p.poll() for p in processes]\n            if any(c is not None and c!=0 for c in codes):raise RuntimeError(f\'{mode} worker failed: {codes}\')\n            if all(c==0 for c in codes):break\n            if time.time()-last_telemetry>30:\n                telemetry=subprocess.run([\'nvidia-smi\',\'--query-gpu=index,uuid,utilization.gpu,memory.used\',\'--format=csv,noheader\'],capture_output=True,text=True)\n                with open(root/\'gpu_telemetry.csv\',\'a\') as stream:\n                    for line in telemetry.stdout.splitlines():stream.write(f\'{time.time()},{line}\\n\')\n                last_telemetry=time.time()\n            time.sleep(5)\n        for gpu in (0,1):assert json.loads((root/f\'worker_{gpu}_complete.json\').read_text())[\'success\']\n    finally:\n        for p in processes:\n            if p.poll() is None:p.terminate()\n        for p in processes:p.wait()\n        for handle in handles:handle.close()\n\n\ndef main():\n    import torch\n    assert torch.cuda.device_count()==2, \'Select GPU T4 x2; no single GPU fallback.\'\n    root=Path(\'/kaggle/working\')/(\'db7_branch_ablation_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\'))\n    root.mkdir()\n    manifest=json.loads(Path(\'protocol.json\').read_text())\n    (root/\'run_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    try:\n        print(\'CONTINUATION: 64 MISSING FITS; BOTH GPU PREFLIGHTS REQUIRED\',flush=True)\n        run_pair(root/\'full\',\'full\')\n        fits=[json.loads(p.read_text()) for p in (root/\'full\').glob(\'S*/seed_*/*/fit_manifest.json\')]\n        actual={(m[\'subject\'],m[\'seed_base\'],m[\'arm\']) for m in fits if m[\'success\'] and m[\'epochs\']==13}\n        assert actual=={tuple(x) for x in manifest[\'remaining_fits\']}\n        (root/\'completion.json\').write_text(json.dumps(dict(success=True,completed_fits=len(actual),source_version=349618438)))\n    except Exception:\n        (root/\'FAILURE.txt\').write_text(traceback.format_exc());raise\n    finally:\n        with zipfile.ZipFile(str(root)+\'.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n            for f in root.rglob(\'*\'):\n                if f.is_file() and f.suffix!=\'.pt\':z.write(f,f.relative_to(root))\n        print(\'OUTPUT:\',root,flush=True)\nif __name__==\'__main__\':main()\n',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'protocol.json').write_text('{"experiment_id": "DB7-016", "subjects": [17, 18, 19, 20], "seeds": [42, 43, 44], "arms": ["W", "S", "I", "WS", "WI", "SI", "WSI"], "full_fits": 64, "smoke_fits": 0, "window_ms": 200, "stride_ms": 10, "trim_ms": 0, "internal_frame_ms": 100, "internal_hop_ms": 50, "train_repetitions": [1, 3, 4, 6], "test_repetitions": [2, 5], "validation": null, "epochs": 13, "batch_size": 512, "dropout": 0.65, "optimizer": "Adam", "weight_decay": 0, "lr_schedule": {"1-3": 0.001, "4-9": 0.0001, "10-13": 1e-05}, "gradient_clip": 5, "seed_rule": "seed_base + 1000*subject", "normalization": "training active segment samples once; spectral train-frame moments", "emg_filter": "per repetition zero-phase fourth-order 20\\u2013450Hz bandpass + 50Hz notch Q30", "modalities": ["EMG12", "ACC36", "gyro36", "mag36"], "checkpoint": "fixed final epoch; only seed42 model weights retained, all seeds retain diagnostics", "gpu_workers": "odd subjects GPU0, even subjects GPU1", "limitations": ["offline annotated segmentation and zero-phase filtering", "fixed test repetitions previously inspected", "subsets differ in parameter count; shared head held fixed", "three seeds are not new independent trials", "bootstrap exploratory, unadjusted pairwise intervals"], "parent_full_fits": 420, "remaining_fits": [[17, 44, "W"], [17, 44, "S"], [17, 44, "I"], [17, 44, "WS"], [17, 44, "WI"], [17, 44, "SI"], [17, 44, "WSI"], [18, 42, "WSI"], [18, 43, "W"], [18, 43, "S"], [18, 43, "I"], [18, 43, "WS"], [18, 43, "WI"], [18, 43, "SI"], [18, 43, "WSI"], [18, 44, "W"], [18, 44, "S"], [18, 44, "I"], [18, 44, "WS"], [18, 44, "WI"], [18, 44, "SI"], [18, 44, "WSI"], [19, 42, "W"], [19, 42, "S"], [19, 42, "I"], [19, 42, "WS"], [19, 42, "WI"], [19, 42, "SI"], [19, 42, "WSI"], [19, 43, "W"], [19, 43, "S"], [19, 43, "I"], [19, 43, "WS"], [19, 43, "WI"], [19, 43, "SI"], [19, 43, "WSI"], [19, 44, "W"], [19, 44, "S"], [19, 44, "I"], [19, 44, "WS"], [19, 44, "WI"], [19, 44, "SI"], [19, 44, "WSI"], [20, 42, "W"], [20, 42, "S"], [20, 42, "I"], [20, 42, "WS"], [20, 42, "WI"], [20, 42, "SI"], [20, 42, "WSI"], [20, 43, "W"], [20, 43, "S"], [20, 43, "I"], [20, 43, "WS"], [20, 43, "WI"], [20, 43, "SI"], [20, 43, "WSI"], [20, 44, "W"], [20, 44, "S"], [20, 44, "I"], [20, 44, "WS"], [20, 44, "WI"], [20, 44, "SI"], [20, 44, "WSI"]], "completed_source_version": 349618438, "continuation_note": "Synthetic preflight on both GPUs retained. Original real-data smoke stage passed. Only missing complete fits retrained from epoch 1; original settings unchanged."}',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'remaining.json').write_text('[[17, 44, "W"], [17, 44, "S"], [17, 44, "I"], [17, 44, "WS"], [17, 44, "WI"], [17, 44, "SI"], [17, 44, "WSI"], [18, 42, "WSI"], [18, 43, "W"], [18, 43, "S"], [18, 43, "I"], [18, 43, "WS"], [18, 43, "WI"], [18, 43, "SI"], [18, 43, "WSI"], [18, 44, "W"], [18, 44, "S"], [18, 44, "I"], [18, 44, "WS"], [18, 44, "WI"], [18, 44, "SI"], [18, 44, "WSI"], [19, 42, "W"], [19, 42, "S"], [19, 42, "I"], [19, 42, "WS"], [19, 42, "WI"], [19, 42, "SI"], [19, 42, "WSI"], [19, 43, "W"], [19, 43, "S"], [19, 43, "I"], [19, 43, "WS"], [19, 43, "WI"], [19, 43, "SI"], [19, 43, "WSI"], [19, 44, "W"], [19, 44, "S"], [19, 44, "I"], [19, 44, "WS"], [19, 44, "WI"], [19, 44, "SI"], [19, 44, "WSI"], [20, 42, "W"], [20, 42, "S"], [20, 42, "I"], [20, 42, "WS"], [20, 42, "WI"], [20, 42, "SI"], [20, 42, "WSI"], [20, 43, "W"], [20, 43, "S"], [20, 43, "I"], [20, 43, "WS"], [20, 43, "WI"], [20, 43, "SI"], [20, 43, "WSI"], [20, 44, "W"], [20, 44, "S"], [20, 44, "I"], [20, 44, "WS"], [20, 44, "WI"], [20, 44, "SI"], [20, 44, "WSI"]]',encoding="utf-8")


In [ ]:
(SCRIPT_DIR/'expected_hashes.json').write_text('{"1": {"train": "199c96549f815ec9ab3c786b2d2b7e3873825bbb06d98171541d77f165fdc3c3", "test": "1b9a16c436bb363b0cbab53a2f7992872ee098477e72a6c79db58c93595a3189"}, "2": {"train": "ebf9afdb857fa7f62bcdccebad65820b63297c855bf985dd4111c0be141adef6", "test": "282556e49de8f9e04761632c99070d089f0334e99266d9a053c3afd8387a9e3b"}, "3": {"train": "39b339484d18d81fe067b40a7011d87d3960f041bfdfeba840ab800a7759236e", "test": "95787b944b3ad970be5fa6710b9f057b4865790df8cd322979d3d5a9da0cda2a"}, "4": {"train": "c06fdfdaf3ff21ec1f1b7b2390a53ee85035b1980dbe307588ac28bdae9ecc75", "test": "e367b625f96355fa1ab86a33bfe2218d8581fc4a3492ca63c39493f8e729aee5"}, "5": {"train": "5349436b0b20efa7e4a603f17704dd65d9069dd6cbee19993c08410e5f7cf2d8", "test": "c90c4489dd1e90f26813ab849a85310761f9ea30ea8c8c28a6f9bc24d6678ff9"}, "6": {"train": "c0800fef039fdf3ac8e559a7ad0e489da618804c25f9603ded3a4a9333601556", "test": "ab47c85d06c375a8e82390ed538b33a528fbf04b618cc675ce66d1634c37a622"}, "7": {"train": "c0eec55ed852c652cdb1e43b7e28405ab3a60eb38888b0478b18d7494275db9c", "test": "c62e58b9e4516671b3725bfd0ee8835d45e15393b47a25cd782b43fc047d1b1f"}, "8": {"train": "e3eebd5db7a347d5c274cb72428b1c29b67d95f7526bfaf0acf11463ea4e16a0", "test": "e6ce2386d91f05851304b45e6a53de3865bfae59e3a84ec0c370881bd9c094e1"}, "9": {"train": "85726c6183171a00afa2b3015ab352f5af84bd663c8b0400465bf657a10ab390", "test": "b473a47d4cde08ae5c913b831f08adf5635f882608402aa081a09d5e54a423f8"}, "10": {"train": "bd7fe887dc1ec80dc487135565bd0b2a63b07f93ee645efc8dde90641ab44a25", "test": "367f8e3f82355890b37302cd47dba6cf0d17042f66b8f62fd6c86c186d949175"}, "11": {"train": "19ff6fba5e41df3fb0c0a378c37419c4489613a5c33b67fd337b2b9b6427ddb4", "test": "ffcdfd0a9e625349c7bcc3a50d547d249c5ad9ca45f4c4453f2fda40c073bc1e"}, "12": {"train": "8453a6058a40cb4c09be341dfacfa8c6201a8415942c516938848d1b3664ef3b", "test": "601cb8fe204398a67482d68b504c6489657b5efffea1cb35033a728aff61fe49"}, "13": {"train": "606967c1fa80ab48fe0b86fd143f77d88477ff586622fb99e075b51e96f4d477", "test": "24086b38c327c2b4502bf72a0b92c79f49637462347b81c93b1acc5165f615ae"}, "14": {"train": "e92b5255d1bc68ec8ace24ad0c7c93d4e5c389cf11169acf195f62193f22d8be", "test": "4eede674cfd0d89ff3dfc9190deb5384ee4ee8e5e791a07aa81d30c3146601b2"}, "15": {"train": "1b6f78205af5d8a8b442dbe4af74ae19579e65469fddc6d8c47756c4cb3fdcd6", "test": "3db3e7b8ab6b26868c75def33b7550084ee1da8e86d4117b3789b5f21047f0f1"}, "16": {"train": "75e0ee9ab6e74fc2a3a73ea12ad21ab1f3f9d3a84e7182bf6301cf797ae6dc60", "test": "3f69e4aabb8826a502986c35b0053cf30253431afc714fd393aa69ddf2822f9f"}, "17": {"train": "575e82bf1c6179518dcec71d223cac920de53cf69c6b804111ea17fd8207a1ff", "test": "7ccf10b647122e897d199f0b0ced829b4b7d1bef5eed908e6d025d4204d92694"}, "18": {"train": "db2b49d660b3ee04597473558ed1445f9539024211d205de1bf34a47eea76273", "test": "5dbf02dc84427a1f0f088b513bfb040a638e533b2cef42a67f4cfe614d99a881"}}',encoding="utf-8")


In [ ]:
subprocess.run([sys.executable,"-u","launch.py"],cwd=SCRIPT_DIR,check=True)
